# Random Forest Regression For Resale Flat Prices

## Imports, Set Ups and Loading of Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import mysql.connector
from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

In [3]:
# Load transform_resale_flat_prices
HDB_db = mysql.connector.connect(
	host='localhost',
	user='bt4301-project',
	passwd='password',
	database='HDB_Data'
)

str_sql = '''
SELECT *
FROM transform_resale_flat_price;
'''

resale_df = pd.read_sql(sql=str_sql, con=HDB_db)

# View resale_df
print(f"Shape of resale_df: {resale_df.shape}")
resale_df.head()

/tmp/ipykernel_10797/537534439.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resale_df = pd.read_sql(sql=str_sql, con=HDB_db)


Shape of resale_df: (223453, 86)


,month_and_year,town,flat_type,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,full_address,month,year,latitude,longitude,planning_area,region,max_floor_lvl,total_dwelling_units,has_market_hawker,has_multistorey_carpark,year_completed,dist_to_nearest_mrt_m,nearest_mrt,n_mrt_within_1km,dist_to_school_m,n_school_within_1km,dist_to_mall_m,n_mall_within_1km,dist_to_food_m,n_food_within_1km,dist_to_park_m,n_park_within_1km,dist_to_supermarket_m,n_supermarket_within_1km,transport_school_pct_bus,transport_school_pct_mrt,transport_school_pct_mrt_bus,transport_school_pct_car,transport_school_pct_mrt_lrt_only,transport_school_pct_mrt_lrt_and_bus,transport_work_pct_bus,transport_work_pct_mrt,transport_work_pct_mrt_bus,transport_work_pct_car,transport_work_pct_mrt_lrt_only,transport_work_pct_mrt_lrt_and_bus,tenancy_pct_owner,tenancy_pct_tenant,dwelling_pct_hdb_1_and_2_room_flats,dwelling_pct_hdb_3_room_flats,dwelling_pct_hdb_4_room_flats,dwelling_pct_hdb_5_room_and_executive_flats,dwelling_pct_condominiums_and_other_apartments,dwelling_pct_landed_properties,dist_to_nearest_carpark_m,nearest_carpark,n_carparks_within_500m,gantry_height,car_park_decks,has_free_parking,is_free_daytime,is_free_halfday,has_short_term_parking,has_night_parking,is_visitor_friendly,has_height_restriction,has_big_vehicle_restriction,has_car_park_basement,dist_to_nearest_bus_stop_m,nearest_bus_stop,n_bus_stop_within_1km,nearest_bus_stop_operating_days_per_week,nearest_bus_stop_busyness_level,building_age,quarter,month_index,town_price_trend_6m,log_resale_price,remaining_lease_years,lease_age,lease_age_sq,price_per_sqm,storey_mid,floor_area_x_storey,storey_ratio,_fp
0,2017-01-01,ANG MO KIO,2 ROOM,10 TO 12,44.0000,Improved,1979,61 years 04 months,"232,000.0000",406 ANG MO KIO AVE 10,1,2017,1.3620,103.8540,ANG MO KIO,NORTH-EAST REGION,12,220,0,0,1978,933.7330,ANG MO KIO MRT STATION,1,282.3040,3,"1,040.7200",0,162.0940,13,715.7270,1,286.8670,2,0.2673,0.0566,0.1635,0.1038,0.0000,0.0000,0.1647,0.1181,0.2649,0.2054,0.0000,0.0000,0.8752,0.1232,0.0895,0.4073,0.2364,0.1038,0.0799,0.0783,"9,526,910.0000",B65L,0,0.0000,0,0,0,0,0,0,0,0,0,0,105.2790,54319,37,7,"1,457.1300",39,1,0,NaN,12.3545,61.3300,37.6700,"1,419.0300","5,272.7300",11.0000,484.0000,0.9167,7c2b5e32315560ed925935df010794f1bae9da78086bcd...
1,2017-01-01,ANG MO KIO,3 ROOM,01 TO 03,67.0000,New Generation,1978,60 years 07 months,"250,000.0000",108 ANG MO KIO AVE 4,1,2017,1.3709,103.8380,ANG MO KIO,NORTH-EAST REGION,12,122,0,0,1978,"1,338.2800",ANG MO KIO MRT STATION,0,402.8780,5,896.3020,1,256.6930,13,638.5520,1,387.0650,2,0.2673,0.0566,0.1635,0.1038,0.0000,0.0000,0.1647,0.1181,0.2649,0.2054,0.0000,0.0000,0.8752,0.1232,0.0895,0.4073,0.2364,0.1038,0.0799,0.0783,"9,526,360.0000",B65L,0,0.0000,0,0,0,0,0,0,0,0,0,0,144.2410,54189,50,7,"1,520.9100",39,1,0,NaN,12.4292,60.5800,38.4200,"1,476.1000","3,731.3400",2.0000,134.0000,0.1667,379e5e2fb6fe56085055cee80bacba3a9a159c3696a0fa...
2,2017-01-01,ANG MO KIO,3 ROOM,07 TO 09,74.0000,New Generation,1978,60 years 04 months,"330,000.0000",118 ANG MO KIO AVE 4,1,2017,1.3733,103.8350,ANG MO KIO,NORTH-EAST REGION,12,151,0,0,1977,"1,451.4900",YIO CHU KANG MRT STATION,0,338.0950,4,"1,231.9300",0,128.6760,11,857.6220,1,513.7650,1,0.2673,0.0566,0.1635,0.1038,0.0000,0.0000,0.1647,0.1181,0.2649,0.2054,0.0000,0.0000,0.8752,0.1232,0.0895,0.4073,0.2364,0.1038,0.0799,0.0783,"9,526,180.0000",B65L,0,0.0000,0,0,0,0,0,0,0,0,0,0,224.3310,56249,42,7,150.2080,40,1,0,NaN,12.7069,60.3300,38.6700,"1,495.3700","4,459.4600",8.0000,592.0000,0.6667,9d3b421e83b92a0e4a5301b697675976f84a14c5952231...
3,2017-01-01,ANG MO KIO,3 ROOM,07 TO 09,67.0000,New Generation,1978,60 years 10 months,"312,000.0000",119 ANG MO KIO AVE 3,1,2017,1.3696,103.8450,ANG MO KIO,NORTH-EAST REGION,11,196,0,0,1977,548.8780,ANG MO KIO MRT STATION,1,369.1510,5,287.9650,4,168.3740,34,597.8800,3,309.5020,5,0.2673,0.0566,0.1635,0.1038,0.0000,0.0000,0.1647,0.1181,0.2649,0.2054,0.0000,0.0000,0.8752,0.1232

In [4]:
# Check resale_df datatypes
print("resale_df info()")
resale_df.info()

resale_df info()
<class 'pandas.DataFrame'>
RangeIndex: 223453 entries, 0 to 223452
Data columns (total 86 columns):
 #   Column                                          Non-Null Count   Dtype         
---  ------                                          --------------   -----         
 0   month_and_year                                  223453 non-null  datetime64[us]
 1   town                                            223453 non-null  str           
 2   flat_type                                       223453 non-null  str           
 3   storey_range                                    223453 non-null  str           
 4   floor_area_sqm                                  223453 non-null  float64       
 5   flat_model                                      223453 non-null  str           
 6   lease_commence_date                             223453 non-null  int64         
 7   remaining_lease                                 223453 non-null  str           
 8   resale_price                

## Random Forest Data Preparation

### Define Feature Groups

In [5]:
# Target
TARGET = "resale_price"

# Columns to drop
drop_cols = [
    "resale_price",          # target
    "log_resale_price",      # leakage (similar to target)
    "price_per_sqm",         # leakage (similar to target)
    "full_address",          # high cardinality
    "_fp",                   # fingerprint
    "nearest_mrt",           # high cardinality
    "nearest_bus_stop",      
    "nearest_carpark",
    "lease_age_sq"           # unnecessary
]

# Numerical Features
num_cols = [
    "floor_area_sqm",
    "storey_mid",
    "max_floor_lvl",
    "total_dwelling_units",
    "latitude",
    "longitude",
    "building_age",
    "remaining_lease_years",

    # distances
    "dist_to_nearest_mrt_m",
    "dist_to_school_m",
    "dist_to_mall_m",
    "dist_to_food_m",
    "dist_to_park_m",
    "dist_to_supermarket_m",
    "dist_to_nearest_bus_stop_m",
    "dist_to_nearest_carpark_m",

    # counts
    "n_mrt_within_1km",
    "n_school_within_1km",
    "n_mall_within_1km",
    "n_food_within_1km",
    "n_park_within_1km",
    "n_supermarket_within_1km",
    "n_bus_stop_within_1km",
    "n_carparks_within_500m",

    # time
    "year",
    "month",
    "quarter",
    "month_index"
]

# Binary Features
binary_cols = [
    "has_market_hawker",
    "has_multistorey_carpark",
    "has_free_parking",
    "has_short_term_parking",
    "has_night_parking",
    # "is_visitor_friendly", # function of has_free_parking, has_short_term_parking, has_night_parking
    "has_big_vehicle_restriction",
    "has_car_park_basement"
]

# Categorical Features
cat_cols = [
    "town",
    "flat_type",
    "storey_range",
    "flat_model",
    "planning_area",
    "region"
]

In [6]:
# Handle potential feature collinearity
# Drop one from each group based on the column list
pct_drop_cols = [
    "transport_school_pct_mrt_lrt_and_bus",
    "transport_work_pct_mrt_lrt_and_bus",
    "tenancy_pct_tenant",
    "dwelling_pct_landed_properties"
]

binary_drop_cols = [
    "is_free_daytime",
    "is_free_halfday",
    "is_visitor_friendly",
    "has_height_restriction"
]

### Preprocessing

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical Pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Binary Pipeline
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

# Categorical Pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine Everything
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("bin", binary_pipeline, binary_cols),
    ("cat", cat_pipeline, cat_cols)
])

### Split the data into train, and test sets

In [8]:
resale_df["month_and_year"] = pd.to_datetime(resale_df["month_and_year"])

# Sort resale_df by date in chronological order
resale_df = resale_df.sort_values(by="month_and_year").reset_index(drop=True)

print("resale_df after sorting:")
resale_df.head()

resale_df after sorting:


,month_and_year,town,flat_type,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,full_address,month,year,latitude,longitude,planning_area,region,max_floor_lvl,total_dwelling_units,has_market_hawker,has_multistorey_carpark,year_completed,dist_to_nearest_mrt_m,nearest_mrt,n_mrt_within_1km,dist_to_school_m,n_school_within_1km,dist_to_mall_m,n_mall_within_1km,dist_to_food_m,n_food_within_1km,dist_to_park_m,n_park_within_1km,dist_to_supermarket_m,n_supermarket_within_1km,transport_school_pct_bus,transport_school_pct_mrt,transport_school_pct_mrt_bus,transport_school_pct_car,transport_school_pct_mrt_lrt_only,transport_school_pct_mrt_lrt_and_bus,transport_work_pct_bus,transport_work_pct_mrt,transport_work_pct_mrt_bus,transport_work_pct_car,transport_work_pct_mrt_lrt_only,transport_work_pct_mrt_lrt_and_bus,tenancy_pct_owner,tenancy_pct_tenant,dwelling_pct_hdb_1_and_2_room_flats,dwelling_pct_hdb_3_room_flats,dwelling_pct_hdb_4_room_flats,dwelling_pct_hdb_5_room_and_executive_flats,dwelling_pct_condominiums_and_other_apartments,dwelling_pct_landed_properties,dist_to_nearest_carpark_m,nearest_carpark,n_carparks_within_500m,gantry_height,car_park_decks,has_free_parking,is_free_daytime,is_free_halfday,has_short_term_parking,has_night_parking,is_visitor_friendly,has_height_restriction,has_big_vehicle_restriction,has_car_park_basement,dist_to_nearest_bus_stop_m,nearest_bus_stop,n_bus_stop_within_1km,nearest_bus_stop_operating_days_per_week,nearest_bus_stop_busyness_level,building_age,quarter,month_index,town_price_trend_6m,log_resale_price,remaining_lease_years,lease_age,lease_age_sq,price_per_sqm,storey_mid,floor_area_x_storey,storey_ratio,_fp
0,2017-01-01,ANG MO KIO,2 ROOM,10 TO 12,44.0000,Improved,1979,61 years 04 months,"232,000.0000",406 ANG MO KIO AVE 10,1,2017,1.3620,103.8540,ANG MO KIO,NORTH-EAST REGION,12,220,0,0,1978,933.7330,ANG MO KIO MRT STATION,1,282.3040,3,"1,040.7200",0,162.0940,13,715.7270,1,286.8670,2,0.2673,0.0566,0.1635,0.1038,0.0000,0.0000,0.1647,0.1181,0.2649,0.2054,0.0000,0.0000,0.8752,0.1232,0.0895,0.4073,0.2364,0.1038,0.0799,0.0783,"9,526,910.0000",B65L,0,0.0000,0,0,0,0,0,0,0,0,0,0,105.2790,54319,37,7,"1,457.1300",39,1,0,NaN,12.3545,61.3300,37.6700,"1,419.0300","5,272.7300",11.0000,484.0000,0.9167,7c2b5e32315560ed925935df010794f1bae9da78086bcd...
1,2017-01-01,CHOA CHU KANG,5 ROOM,04 TO 06,123.0000,Improved,1993,75 years 05 months,"435,000.0000",248 CHOA CHU KANG AVE 2,1,2017,1.3790,103.7460,CHOA CHU KANG,WEST REGION,18,114,0,0,1992,164.4330,SOUTH VIEW LRT STATION,5,288.0670,4,303.3580,5,239.5950,34,885.7580,1,248.2170,6,0.1542,0.1012,0.2795,0.0530,0.0000,0.0000,0.0935,0.1686,0.2713,0.1994,0.0000,0.0000,0.9635,0.0304,0.0324,0.0324,0.4251,0.3887,0.1012,0.0182,"9,527,900.0000",B65L,0,0.0000,0,0,0,0,0,0,0,0,0,0,173.6620,44441,71,7,385.9210,25,1,0,NaN,12.9831,75.4200,23.5800,556.0160,"3,536.5900",5.0000,615.0000,0.2778,a02f455c049e522753ab3ac4bef06854078803971d3a71...
2,2017-01-01,CHOA CHU KANG,4 ROOM,01 TO 03,102.0000,Model A,1996,78 years 06 months,"343,000.0000",771 CHOA CHU KANG ST 54,1,2017,1.3943,103.7490,CHOA CHU KANG,WEST REGION,16,92,0,0,1995,404.0300,YEW TEE MRT STATION,1,152.2110,5,401.4690,2,355.0760,17,688.1570,2,357.1460,4,0.1542,0.1012,0.2795,0.0530,0.0000,0.0000,0.0935,0.1686,0.2713,0.1994,0.0000,0.0000,0.9635,0.0304,0.0324,0.0324,0.4251,0.3887,0.1012,0.0182,"9,526,170.0000",B65L,0,0.0000,0,0,0,0,0,0,0,0,0,0,218.7130,45359,50,7,229.7460,22,1,0,NaN,12.7455,78.5000,20.5000,420.2500,"3,362.7500",2.0000,204.0000,0.1250,cc6269e171d1a97572131d45ec5ba9e88a572cdc2dd560...
3,2017-01-01,CHOA CHU KANG,4 ROOM,01 TO 03,90.0000,Model A,2003,85 years 01 month,"325,000.0000",692B CHOA CHU KANG CRES,1,2017,1.4007,103.7510,CHOA CHU KANG,WEST REGION,24,188,0,0,2001,531.0240,YEW TEE MRT STATION,1,412.1560,4,550.3520,3,108.1970,14,920.7960,1,570.4560,2,0.1542,0.1012,0.2795,0.0530,0.0000,0.0000,0.0935,0.1686,0.2713,0.1994,0.0000,0.0000,0.9635,0.0304,0.0324,0.0324,0.4251,0.3887,0.1012,0.0182,

In [9]:
# Split the dataset into train and test
split_point = int(len(resale_df) * 0.8)

train_df = resale_df.iloc[:split_point]
test_df = resale_df.iloc[split_point:]

In [10]:
# Drop the necessary columns

X_train = train_df.drop(columns=drop_cols + pct_drop_cols + binary_drop_cols, errors="ignore")
y_train = train_df[TARGET]

X_test = test_df.drop(columns=drop_cols + pct_drop_cols + binary_drop_cols, errors="ignore")
y_test = test_df[TARGET]

In [11]:
X_train.shape

(178762, 69)

In [12]:
X_test.shape

(44691, 69)

## Baseline Model

### Train the model

In [13]:
model = Pipeline([
    ("preprocessing", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

In [14]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

In [15]:
# View feature importance

rf_model = model.named_steps["rf"]
feature_names = model.named_steps["preprocessing"].get_feature_names_out()

import pandas as pd
feat_imp = pd.Series(rf_model.feature_importances_, index=feature_names)

# View top features
print(feat_imp.sort_values(ascending=False).head(20))

num__floor_area_sqm              0.4541
cat__region_CENTRAL REGION       0.1569
num__month_index                 0.1273
num__max_floor_lvl               0.0739
num__remaining_lease_years       0.0394
num__latitude                    0.0252
num__longitude                   0.0232
num__building_age                0.0185
num__storey_mid                  0.0105
cat__flat_model_DBSS             0.0075
num__dist_to_nearest_carpark_m   0.0067
num__dist_to_nearest_mrt_m       0.0064
num__n_food_within_1km           0.0059
num__dist_to_park_m              0.0048
num__dist_to_mall_m              0.0043
cat__flat_model_Model A          0.0026
num__n_mrt_within_1km            0.0022
num__n_mall_within_1km           0.0021
num__year                        0.0019
cat__flat_type_4 ROOM            0.0018
dtype: float64


### Evaluate

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 61459.943309163966
RMSE: 78201.67894468522


## Improved model -- using log transform

In [17]:
from sklearn.compose import TransformedTargetRegressor

random_forest = RandomForestRegressor(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

improved_model = Pipeline([
    ("preprocessing", preprocessor),
    ("rf", TransformedTargetRegressor(
        regressor=random_forest,
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

improved_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

In [18]:
y_pred_improved = improved_model.predict(X_test)

mae_improved = mean_absolute_error(y_test, y_pred_improved)
rmse_improved = np.sqrt(mean_squared_error(y_test, y_pred_improved))

print("MAE:", mae_improved)
print("RMSE:", rmse_improved)

MAE: 62877.90634336497
RMSE: 80230.7149811889


## Hyperparameter Tuning

In [19]:
param_dist = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_depth": [12, 20, 30, 50],
    "rf__min_samples_leaf": [1, 2, 5]
}

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [ ]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [18]:
best_model = random_search.best_estimator_

print("Best model:", best_model)
print("Best params:", random_search.best_params_)

Best model: Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['floor_area_sqm',
                                                   'storey_mid',
                                                   'max_floor_lvl',
                                                   'total_dwelling_units',
                                                   'latitude', 'longitude',
                                                   'building_age',
                                                   'remaining_lease_years',
                                                   'dist_to_nearest_mrt_m',
                                                   'dist_to_school_m',
                                                   'dist_to_mall_m',
   

In [19]:
y_pred = best_model.predict(X_test)

In [20]:
mae_tuned = mean_absolute_error(y_test, y_pred)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae_tuned)
print("RMSE:", rmse_tuned)

MAE: 76690.7672488524
RMSE: 98915.02270272114


In [21]:
HDB_db.close()